# Accuracy Assessment

This notebook evaluates the building-level classification results using an independent ground-truth dataset.

For each reference building, the percentage of pixels classified as asbestos is calculated. These percentages are then used to:

1. evaluate different decision thresholds;
2. generate the Precision-Recall Curve;
3. compute the final accuracy metrics for a selected threshold.

### Input

- Classified raster
- Ground-truth building polygons

### Output

- Building-level asbestos percentages
- Accuracy metrics
- Confusion matrix

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.mask

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
)


# Input and output files

data_dir = Path("../output")
validation_dir = Path("../data")

classification_path = data_dir / "MLC_classification.tif"

# Positive ground-truth buildings
asbestos_buildings = validation_dir / "asbestos_buildings.shp"

# One or more shapefiles containing negative ground-truth buildings
non_asbestos_buildings = [
    validation_dir / "non_asbestos_buildings.shp",
]

# Field containing the ground-truth label (only needed if already present)
label_field = "y_true"



# User parameters
# Integer value corresponding to the asbestos class
asbestos_class = 1

# Decision threshold used for the final accuracy assessment
decision_threshold = 0.30


# Load ground-truth data

asbestos = gpd.read_file(asbestos_buildings)

gdfs = []
for path in non_asbestos_buildings:
    gdfs.append(gpd.read_file(path))

non_asbestos = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs,
)

asbestos["y_true"] = 1
non_asbestos["y_true"] = 0

ground_truth = gpd.GeoDataFrame(
    pd.concat([asbestos, non_asbestos], ignore_index=True),
    crs=asbestos.crs,
)


# Read classified raster

with rasterio.open(classification_path) as raster:

    if ground_truth.crs != raster.crs:
        ground_truth = ground_truth.to_crs(raster.crs)

    asbestos_percent = []

    for geom in ground_truth.geometry:

        out_img, _ = rasterio.mask.mask(
            raster,
            [geom],
            crop=True,
        )

        arr = out_img[0]

        # Remove NoData pixels
        arr = arr[arr != raster.nodata]

        if len(arr) == 0:
            asbestos_percent.append(0)
            continue

        percent = np.sum(arr == asbestos_class) / len(arr)

        asbestos_percent.append(percent)


# Prepare validation data

ground_truth["asbestos_percent"] = asbestos_percent

y_true = ground_truth["y_true"].values

scores = ground_truth["asbestos_percent"].values



# Evaluate different decision thresholds

thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:

    y_pred = (scores >= threshold).astype(int)

    precision = precision_score(y_true, y_pred)

    recall = recall_score(y_true, y_pred)

    f1 = f1_score(y_true, y_pred)

    print(
        f"Threshold {int(threshold*100)}% "
        f"→ Precision={precision:.2f}, "
        f"Recall={recall:.2f}, "
        f"F1={f1:.2f}"
    )


# Final accuracy assessment

y_pred = (scores >= decision_threshold).astype(int)

accuracy = accuracy_score(y_true, y_pred)

conf_matrix = confusion_matrix(y_true, y_pred)

report = classification_report(
    y_true,
    y_pred,
    target_names=["Non-asbestos", "Asbestos"],
    digits=4,
)


# Display results

print(f"Accuracy: {accuracy:.4f}")

print("\nConfusion matrix:")

print(conf_matrix)

print("\nClassification report:")

print(report)